# Travnr ML — Feature Engineering Audit & Analysis

**Goal**: Extract production-aligned features (matching `featureExtractor.ts`), audit each feature's quality, check data sources, and produce a report.

**Key questions**:
1. Does each feature correctly mirror what `featureExtractor.ts` extracts?
2. Is the underlying data in the database correct and complete?
3. Are the temporal/journey-tracking features viable?
4. What's broken, and what needs fixing?

In [1]:
!pip install xgboost pandas numpy scikit-learn matplotlib seaborn --quiet
import pandas as pd
import numpy as np
import json
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
print('All packages loaded')

All packages loaded


In [2]:
# ──────────────────────────────────────────────
# CELL: Load training_data.csv
# ──────────────────────────────────────────────
df = pd.read_csv('training_data.csv')\
    .sort_values(by=['monitored_flight_id', 'scored_at'])\
    .reset_index(drop=True)
print(f'Loaded {len(df)} rows')
print(f'Columns: {list(df.columns)}\n')
print(f'Date range: {df["scored_at"].min()} to {df["scored_at"].max()}')
print(f'Unique flights: {df["monitored_flight_id"].nunique()}')
print(f'Unique carriers: {df["carrier_iata"].nunique()}')
print(f'Unique routes: {(df["origin_iata"] + "->" + df["destination_iata"]).nunique()}')

Loaded 7258 rows
Columns: ['id', 'monitored_flight_id', 'heuristic_score', 'signals', 'tail_number', 'equipment_type', 'scored_at', 'tier', 'flight_number', 'carrier_iata', 'origin_iata', 'destination_iata', 'departure_date', 'departure_time', 'resolved_status', 'resolved_delay_minutes', 'departure_hour', 'day_of_week', 'month']

Date range: 2026-05-17 21:33:41.424572 to 2026-06-10 23:23:28.9824
Unique flights: 478
Unique carriers: 26
Unique routes: 312


In [3]:
# ──────────────────────────────────────────────
# CELL: Parse signals JSON and audit data sources
# ──────────────────────────────────────────────
def parse_json(x):
    if isinstance(x, str):
        return json.loads(x)
    return x

signals_list = df['signals'].apply(lambda s: parse_json(s))

# Check top-level fields present
def field_coverage(sig_list, prefix=''):
    total = len(sig_list)
    coverage = {}
    for sig in sig_list:
        for k in sig:
            coverage[k] = coverage.get(k, 0) + 1
    print(f'--- {prefix}Top-level field coverage ({total} rows) ---')
    for k, v in sorted(coverage.items(), key=lambda x: -x[1]):
        print(f'  {k:30s}: {v:6d}/{total} ({v/total*100:5.1f}%)')
    return coverage

field_coverage(signals_list)

# Check specific sub-objects
print()
def subfield_coverage(sig_list, parent_key, total):
    count = 0
    keys = {}
    for sig in sig_list:
        obj = sig.get(parent_key)
        if isinstance(obj, dict):
            count += 1
            for k in obj:
                keys[k] = keys.get(k, 0) + 1
    print(f'--- {parent_key} field coverage ({count}/{total} rows with data) ---')
    for k, v in sorted(keys.items(), key=lambda x: -x[1]):
        print(f'  {k:30s}: {v:6d}/{count} ({v/count*100:5.1f}% of present)')

total = len(signals_list)
for parent in ['originWeather', 'destinationWeather', 'flightStatus', 'carrierHealth', 'nasOrigin', 'nasDestination']:
    subfield_coverage(signals_list, parent, total)
    print()

--- Top-level field coverage (7258 rows) ---
  signals                       :   7258/7258 (100.0%)
  cancelled                     :   7258/7258 (100.0%)
  flightStatus                  :   7258/7258 (100.0%)
  originWeather                 :   7258/7258 (100.0%)
  destinationWeather            :   7258/7258 (100.0%)
  horizon                       :   7239/7258 ( 99.7%)
  nasOrigin                     :   7239/7258 ( 99.7%)
  carrierHealth                 :   7239/7258 ( 99.7%)
  nasDestination                :   7239/7258 ( 99.7%)
  hoursUntilDeparture           :   7239/7258 ( 99.7%)
  simulated                     :      2/7258 (  0.0%)
  simulatedReason               :      2/7258 (  0.0%)

--- originWeather field coverage (7258/7258 rows with data) ---
  hasFreezing                   :   7258/7258 (100.0% of present)
  flightCategory                :   7258/7258 (100.0% of present)
  hasThunderstorm               :   7258/7258 (100.0% of present)
  ceilingFt                     

---
## Phase 1: Data Quality Audit

Before extracting features, let's check the raw data quality.

In [4]:
# ──────────────────────────────────────────────
# CELL: Cancellation label extraction
# ──────────────────────────────────────────────
def is_cancelled(sig):
    fs = sig.get('flightStatus', {})
    if isinstance(fs, str):
        try: fs = json.loads(fs)
        except: fs = {}
    if isinstance(fs, dict):
        cancelled = fs.get('cancelled', False)
        status = fs.get('status', '')
        return cancelled or (isinstance(status, str) and status.lower().startswith('cancel'))
    return False

y_cycle = pd.Series(signals_list.apply(lambda s: int(is_cancelled(s))), name='cancelled')

print('--- Cancellation Labels ---')
print(f'Total cycles: {len(y_cycle)}')
print(f'Cancelled: {y_cycle.sum()} ({y_cycle.mean()*100:.2f}%)')
print(f'Arrived:   {(1-y_cycle).sum()} ({(1-y_cycle).mean()*100:.2f}%)')

# Filter post-cancellation cycles
def filter_post_cancellation(df, y):
    mask = pd.Series(True, index=df.index)
    for fid in df['monitored_flight_id'].unique():
        flight_idx = df[df['monitored_flight_id'] == fid].index
        canc_idx = flight_idx[y.loc[flight_idx] == 1]
        if len(canc_idx) > 0:
            mask[flight_idx[flight_idx > canc_idx[0]]] = False
    return mask

valid_mask = filter_post_cancellation(df, y_cycle)
y_filtered = y_cycle[valid_mask].reset_index(drop=True)
print(f'\nAfter post-cancellation filter:')
print(f'  Removed: {(~valid_mask).sum()} rows')
print(f'  Positive: {y_filtered.sum()} / {len(y_filtered)} ({y_filtered.mean()*100:.2f}%)')

--- Cancellation Labels ---
Total cycles: 7258
Cancelled: 47 (0.65%)
Arrived:   7211 (99.35%)

After post-cancellation filter:
  Removed: 30 rows
  Positive: 17 / 7228 (0.24%)


In [5]:
# ──────────────────────────────────────────────
# CELL: Simulated / invalid data check
# ──────────────────────────────────────────────
simulated_count = signals_list.apply(lambda s: s.get('simulated', False)).sum()
print(f'Simulated cycles: {simulated_count}')

# Check cancellationRate24h distribution in raw data
canc_rates = []
for sig in signals_list:
    ch = sig.get('carrierHealth', {})
    if isinstance(ch, dict):
        canc_rates.append(ch.get('cancellationRate24h', 0) or 0)
cr = pd.Series(canc_rates)
print(f'\nCancellationRate24h (raw from DB):')
print(f'  Range: {cr.min():.6f} to {cr.max():.6f}')
print(f'  Mean:  {cr.mean():.6f}')
print(f'  Std:   {cr.std():.6f}')
print(f'  Nonzero: {(cr>0).sum()}/{len(cr)} ({(cr>0).mean()*100:.1f}%)')
print(f'  NOTE: This is a RATE (0-1), NOT a percentage!')
print(f'  featureExtractor.ts uses it as-is (correct)')
print(f'  T.ipynb/T2.ipynb divide by 100 (BUG: makes it 0-0.01)')

Simulated cycles: 2

CancellationRate24h (raw from DB):
  Range: 0.000000 to 0.280000
  Mean:  0.001719
  Std:   0.012357
  Nonzero: 978/7258 (13.5%)
  NOTE: This is a RATE (0-1), NOT a percentage!
  featureExtractor.ts uses it as-is (correct)
  T.ipynb/T2.ipynb divide by 100 (BUG: makes it 0-0.01)


In [6]:
# ──────────────────────────────────────────────
# CELL: Check flightStatus delay data — is there ANY?
# ──────────────────────────────────────────────
delay_minutes = []
inbound_delays = []
for sig in signals_list:
    fs = sig.get('flightStatus', {})
    if isinstance(fs, dict):
        delay_minutes.append(fs.get('delayMinutes', 0) or 0)
        inbound_delays.append(fs.get('inboundDelayMinutes', 0) or 0)

dm = pd.Series(delay_minutes)
idm = pd.Series(inbound_delays)

print('--- flightStatus delayMinutes ---')
print(f'  Non-zero: {(dm>0).sum()}/{len(dm)} ({((dm>0).mean()*100):.2f}%)')
print(f'  Unique values: {dm.nunique()}')
print(f'  Max: {dm.max()}')
print(f'  These are the AeroDataBox delay readings (mostly 0)')

--- flightStatus delayMinutes ---
  Non-zero: 2/7199 (0.03%)
  Unique values: 2
  Max: 90
  These are the AeroDataBox delay readings (mostly 0)


In [7]:
# ──────────────────────────────────────────────
# CELL: Check NAS avgDelayMinutes — is there ANY?
# ──────────────────────────────────────────────
nas_delays_o = []
nas_delays_d = []
for sig in signals_list:
    no = sig.get('nasOrigin', {})
    nd = sig.get('nasDestination', {})
    if isinstance(no, dict):
        nas_delays_o.append(no.get('avgDelayMinutes', 0) or 0)
    if isinstance(nd, dict):
        nas_delays_d.append(nd.get('avgDelayMinutes', 0) or 0)

nas_o = pd.Series(nas_delays_o)
nas_d = pd.Series(nas_delays_d)

print('--- NAS avgDelayMinutes ---')
print(f'  nasOrigin non-zero:  {(nas_o>0).sum()}/{len(nas_o)} ({((nas_o>0).mean()*100):.1f}%)')
print(f'  nasDestination non-zero: {(nas_d>0).sum()}/{len(nas_d)} ({((nas_d>0).mean()*100):.1f}%)')
print(f'  nasOrigin unique values: {nas_o.unique()[:20]}')
nas_max = pd.concat([nas_o, nas_d], axis=1).max(axis=1)
print(f'  Combined max (atc_avg_delay) non-zero: {(nas_max>0).sum()}/{len(nas_max)} ({((nas_max>0).mean()*100):.1f}%)')
print(f'\n  NOTE: featureExtractor.ts uses atc_avg_delay = max(nasOrigin, nasDestination)')
print(f'  T.ipynb/T2.ipynb use flightStatus.delayMinutes instead (WRONG source)')

--- NAS avgDelayMinutes ---
  nasOrigin non-zero:  636/7258 (8.8%)
  nasDestination non-zero: 328/7258 (4.5%)
  nasOrigin unique values: [  0 176 112 155  49  78  80  27 284 109  62]
  Combined max (atc_avg_delay) non-zero: 925/7258 (12.7%)

  NOTE: featureExtractor.ts uses atc_avg_delay = max(nasOrigin, nasDestination)
  T.ipynb/T2.ipynb use flightStatus.delayMinutes instead (WRONG source)


In [8]:
# ──────────────────────────────────────────────
# CELL: Destination weather — field availability
# ──────────────────────────────────────────────
dst_vis = []
for sig in signals_list:
    dw = sig.get('destinationWeather', {})
    if isinstance(dw, dict):
        dst_vis.append(dw.get('visibilityMiles', 'MISSING'))
    else:
        dst_vis.append('MISSING')

dst_vis_s = pd.Series(dst_vis)
unique_vis = dst_vis_s.value_counts()
print('--- destinationWeather.visibilityMiles ---')
print(f'  Unique values:\n{unique_vis}')
print(f'\n  NOTE: monitor.ts strips destinationWeather to only 3 fields:')
print(f'  flightCategory, hasThunderstorm, hasFreezing')
print(f'  destination_visibility_miles in T.ipynb is ALWAYS 10.0 (default)')

--- destinationWeather.visibilityMiles ---
  Unique values:
MISSING    7258
Name: count, dtype: int64

  NOTE: monitor.ts strips destinationWeather to only 3 fields:
  flightCategory, hasThunderstorm, hasFreezing
  destination_visibility_miles in T.ipynb is ALWAYS 10.0 (default)


In [9]:
# ──────────────────────────────────────────────
# CELL: Journey tracking — checkpoints per flight
# ──────────────────────────────────────────────
cp_counts = df.groupby('monitored_flight_id').size()
print('--- Checkpoint distribution per flight ---')
print(f'  Min checkpoints: {cp_counts.min()}')
print(f'  Max checkpoints: {cp_counts.max()}')
print(f'  Mean checkpoints: {cp_counts.mean():.1f}')
print(f'  Median checkpoints: {cp_counts.median():.0f}')
print(f'\n  Distribution:')
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'    {p}th percentile: {cp_counts.quantile(p/100):.0f}')

# Check how many flights have delays across checkpoints
print(f'\n--- Temporal feature viability ---')
print(f'  Delay values exist: {(dm>0).sum()} out of {len(dm)} ({((dm>0).mean()*100):.1f}%)')
print(f'  With 99.9% zeros, delay_delta_30m and delay_rolling_avg_4 are effectively dead')
print(f'  The temporal tracking logic works, but the delay INPUT is missing')

--- Checkpoint distribution per flight ---
  Min checkpoints: 1
  Max checkpoints: 68
  Mean checkpoints: 15.2
  Median checkpoints: 16

  Distribution:
    10th percentile: 1
    25th percentile: 2
    50th percentile: 16
    75th percentile: 26
    90th percentile: 31
    95th percentile: 31
    99th percentile: 31

--- Temporal feature viability ---
  Delay values exist: 2 out of 7199 (0.0%)
  With 99.9% zeros, delay_delta_30m and delay_rolling_avg_4 are effectively dead
  The temporal tracking logic works, but the delay INPUT is missing


---
## Phase 2: Production-Aligned Feature Extraction

This extracts the **exact 25 features** that `featureExtractor.ts` produces, using the **same data sources** and **same scaling**.

In [10]:
# ──────────────────────────────────────────────
# CELL: Feature extraction — matching featureExtractor.ts
# ──────────────────────────────────────────────
flight_cat_map = {'VFR': 0, 'MVFR': 1, 'IFR': 2, 'LIFR': 3}

def extract_production_features(df):
    """Extract features EXACTLY matching featureExtractor.ts
    
    Sources:
      - flightStatus: inbound_delay, current_delay
      - nasOrigin/nasDestination: atc_avg_delay (NOT flightStatus.delayMinutes)
      - originWeather: flight_cat, thunderstorm, freezing, wind, gust, visibility, ceiling
      - destinationWeather: flight_cat, thunderstorm, freezing (ONLY what's stored)
      - carrierHealth: cancellation_rate, avg_delay, sample_size
      - schedule: departure_hour, day_of_week, month
      - derived: is_weekend, is_rush_hour, is_summer
      - temporal: delay_delta_30m, delay_rolling_avg_4, checkpoint_number
    """
    features = []
    flight_history = {}
    
    for idx, row in df.iterrows():
        fid = row['monitored_flight_id']
        sig = parse_json(row['signals'])
        
        # Parse sub-objects
        fs = sig.get('flightStatus', {})
        if isinstance(fs, str): fs = json.loads(fs) if fs else {}
        if not isinstance(fs, dict): fs = {}
        
        ow = sig.get('originWeather', {})
        if isinstance(ow, str): ow = json.loads(ow)
        if not isinstance(ow, dict): ow = {}
        
        dw = sig.get('destinationWeather', {})
        if isinstance(dw, str): dw = json.loads(dw)
        if not isinstance(dw, dict): dw = {}
        
        no = sig.get('nasOrigin', {})
        if isinstance(no, str): no = json.loads(no)
        if not isinstance(no, dict): no = {}
        
        nd = sig.get('nasDestination', {})
        if isinstance(nd, str): nd = json.loads(nd)
        if not isinstance(nd, dict): nd = {}
        
        ch = sig.get('carrierHealth', {})
        if isinstance(ch, str): ch = json.loads(ch)
        if not isinstance(ch, dict): ch = {}
        
        # Flight schedule info from DataFrame columns
        carrier_iata = row['carrier_iata']
        flight_number = row['flight_number']
        departure_date = row['departure_date']
        scored_at = pd.to_datetime(row['scored_at'], utc=True)
        scheduled_dep = pd.to_datetime(fs.get('departureTime', row['departure_time'] or '2026-01-01'), utc=True)
        
        # --- 1-2: AeroDataBox flight status ---
        inbound_delay = fs.get('inboundDelayMinutes', 0) or 0
        current_delay = fs.get('delayMinutes', 0) or 0
        
        # --- 3: FAA NAS (atc_avg_delay from NAS, NOT flightStatus) ---
        no_delay = no.get('avgDelayMinutes', 0) or 0
        nd_delay = nd.get('avgDelayMinutes', 0) or 0
        atc_avg_delay = max(no_delay, nd_delay)
        
        # --- 4-10: Origin weather ---
        origin_flight_cat = flight_cat_map.get(ow.get('flightCategory', 'VFR'), 0)
        origin_thunderstorm = 1 if ow.get('hasThunderstorm', False) else 0
        origin_freezing = 1 if ow.get('hasFreezing', False) else 0
        origin_wind_speed = ow.get('windSpeedKt', 0) or 0
        origin_gust_speed = ow.get('gustSpeedKt', 0) or 0
        origin_visibility = ow.get('visibilityMiles', 10) or 10
        origin_ceiling = ow.get('ceilingFt', 99999) or 99999
        
        # --- 11-13: Destination weather (only fields stored in DB) ---
        dest_flight_cat = flight_cat_map.get(dw.get('flightCategory', 'VFR'), 0)
        dest_thunderstorm = 1 if dw.get('hasThunderstorm', False) else 0
        dest_freezing = 1 if dw.get('hasFreezing', False) else 0
        
        # --- 14-16: Carrier health (correct scaling: NOT divided by 100) ---
        carrier_cancel_rate = ch.get('cancellationRate24h', 0) or 0
        carrier_avg_delay = ch.get('avgDelay24h', 0) or 0.0
        carrier_sample_size = ch.get('sampleSize', 0) or 0
        
        # --- 17-19: Flight schedule ---
        if pd.notna(scheduled_dep):
            departure_hour = scheduled_dep.hour
            day_of_week = (scheduled_dep.dayofweek + 1) % 7  # JS DOW (Sun=0) to match featureExtractor.ts
            month = scheduled_dep.month
        else:
            departure_hour = int(row.get('departure_hour', 0) or 0)
            day_of_week = (int(row.get('day_of_week', 0) or 0) + 1) % 7  # Convert CSV Python DOW to JS DOW
            month = int(row.get('month', 1) or 1)
        
        # --- 20-22: Derived booleans ---
        is_weekend = 1 if day_of_week in [0, 6] else 0  # JS DOW: Sun=0, Sat=6
        is_rush_hour = 1 if departure_hour in [7, 8, 9, 16, 17, 18] else 0
        is_summer = 1 if month in [6, 7, 8] else 0
        
        # --- 23-25: Temporal features (same logic as featureExtractor.ts) ---
        flight_key = f'{carrier_iata}-{flight_number}-{departure_date}'
        hist = flight_history.get(flight_key)
        if hist is None:
            hist = {'prev_delay': 0, 'recent_delays': [], 'checkpoint_count': 0}
            flight_history[flight_key] = hist
        hist['checkpoint_count'] += 1
        
        # delay_delta (current - previous)
        delay_delta_30m = current_delay - hist['prev_delay']
        hist['prev_delay'] = current_delay
        
        # rolling average of last 4
        hist['recent_delays'].append(current_delay)
        if len(hist['recent_delays']) > 4:
            hist['recent_delays'].pop(0)
        delay_rolling_avg_4 = sum(hist['recent_delays']) / max(len(hist['recent_delays']), 1)
        
        checkpoint_number = hist['checkpoint_count']
        
        features.append({
            'inbound_delay': inbound_delay,
            'current_delay': current_delay,
            'atc_avg_delay': atc_avg_delay,
            'origin_flight_cat': origin_flight_cat,
            'origin_thunderstorm': origin_thunderstorm,
            'origin_freezing': origin_freezing,
            'origin_wind_speed': origin_wind_speed,
            'origin_gust_speed': origin_gust_speed,
            'origin_visibility': origin_visibility,
            'origin_ceiling': origin_ceiling,
            'dest_flight_cat': dest_flight_cat,
            'dest_thunderstorm': dest_thunderstorm,
            'dest_freezing': dest_freezing,
            'carrier_cancel_rate': carrier_cancel_rate,
            'carrier_avg_delay': carrier_avg_delay,
            'carrier_sample_size': carrier_sample_size,
            'departure_hour': departure_hour,
            'day_of_week': day_of_week,
            'month': month,
            'is_weekend': is_weekend,
            'is_rush_hour': is_rush_hour,
            'is_summer': is_summer,
            'delay_delta_30m': delay_delta_30m,
            'delay_rolling_avg_4': delay_rolling_avg_4,
            'checkpoint_number': checkpoint_number,
        })
    
    return pd.DataFrame(features)

X_prod = extract_production_features(df)
X_prod_filtered = X_prod[valid_mask.values].reset_index(drop=True)
y_final = y_filtered

print(f'Production features: {len(X_prod.columns)} features')
print(f'  Feature names: {list(X_prod.columns)}\n')
print(f'  Unfiltered: {len(X_prod)} rows')
print(f'  After label-leakage filter: {len(X_prod_filtered)} rows')
print(f'  Positive labels: {y_final.sum()} ({y_final.mean()*100:.2f}%)')

Production features: 25 features
  Feature names: ['inbound_delay', 'current_delay', 'atc_avg_delay', 'origin_flight_cat', 'origin_thunderstorm', 'origin_freezing', 'origin_wind_speed', 'origin_gust_speed', 'origin_visibility', 'origin_ceiling', 'dest_flight_cat', 'dest_thunderstorm', 'dest_freezing', 'carrier_cancel_rate', 'carrier_avg_delay', 'carrier_sample_size', 'departure_hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour', 'is_summer', 'delay_delta_30m', 'delay_rolling_avg_4', 'checkpoint_number']

  Unfiltered: 7258 rows
  After label-leakage filter: 7228 rows
  Positive labels: 17 (0.24%)


In [11]:
# ──────────────────────────────────────────────
# CELL: Validate NaN/Inf and basic stats
# ──────────────────────────────────────────────
null_counts = X_prod_filtered.isnull().sum()
inf_counts = X_prod_filtered.isin([float('inf'), float('-inf')]).sum()
print('--- Sanity Check ---')
if null_counts.sum() == 0 and inf_counts.sum() == 0:
    print('[PASS] No NaN or Inf values')
else:
    print('[WARN] Issues found:')
    for c in X_prod_filtered.columns:
        if null_counts[c] > 0: print(f'  {c}: {null_counts[c]} nulls')
        if inf_counts[c] > 0: print(f'  {c}: {inf_counts[c]} infs')

print(f'\n--- Feature Ranges ---')
for c in X_prod_filtered.columns:
    vals = X_prod_filtered[c]
    print(f'  {c:25s}: {vals.min():>10.4f} to {vals.max():>10.4f} (mean={vals.mean():>10.4f}, std={vals.std():>10.4f})')

--- Sanity Check ---
[PASS] No NaN or Inf values

--- Feature Ranges ---
  inbound_delay            :     0.0000 to    90.0000 (mean=    0.0249, std=    1.4970)
  current_delay            :     0.0000 to    90.0000 (mean=    0.0249, std=    1.4970)
  atc_avg_delay            :     0.0000 to   284.0000 (mean=   18.9830, std=   61.6732)
  origin_flight_cat        :     0.0000 to     3.0000 (mean=    0.1547, std=    0.3914)
  origin_thunderstorm      :     0.0000 to     1.0000 (mean=    0.0311, std=    0.1737)
  origin_freezing          :     0.0000 to     0.0000 (mean=    0.0000, std=    0.0000)
  origin_wind_speed        :     0.0000 to    24.0000 (mean=    7.6235, std=    4.6992)
  origin_gust_speed        :     0.0000 to    37.0000 (mean=    4.4232, std=    8.9931)
  origin_visibility        :     0.5000 to    10.0000 (mean=    9.7046, std=    1.1251)
  origin_ceiling           :   500.0000 to 99999.0000 (mean=42907.4574, std=42238.4631)
  dest_flight_cat          :     0.0000 to     

In [12]:
# ──────────────────────────────────────────────
# CELL: Verify exact match with featureExtractor.ts ORDER
# ──────────────────────────────────────────────
expected_order = [
    'inbound_delay', 'current_delay',           # 1-2: AeroDataBox
    'atc_avg_delay',                            # 3: FAA NAS
    'origin_flight_cat', 'origin_thunderstorm', 'origin_freezing',  # 4-6
    'origin_wind_speed', 'origin_gust_speed', 'origin_visibility', 'origin_ceiling',  # 7-10
    'dest_flight_cat', 'dest_thunderstorm', 'dest_freezing',  # 11-13
    'carrier_cancel_rate', 'carrier_avg_delay', 'carrier_sample_size',  # 14-16
    'departure_hour', 'day_of_week', 'month',   # 17-19
    'is_weekend', 'is_rush_hour', 'is_summer',  # 20-22
    'delay_delta_30m', 'delay_rolling_avg_4', 'checkpoint_number',  # 23-25
]

assert list(X_prod_filtered.columns) == expected_order, \
    f'Feature order mismatch!\nGot:      {list(X_prod_filtered.columns)}\nExpected: {expected_order}'
print('[PASS] Feature count and order match featureExtractor.ts exactly (25 features)')

[PASS] Feature count and order match featureExtractor.ts exactly (25 features)


---
## Phase 3: Per-Feature Deep Analysis

For each feature, we analyze:
1. **Data source** — where does it come from?
2. **Distribution** — range, spread, zeros
3. **Label correlation** — cancelled vs arrived means
4. **Signal quality** — is it predictive?
5. **Data integrity** — is the source data correct?

In [13]:
# ──────────────────────────────────────────────
# CELL: Feature analysis report generator
# ──────────────────────────────────────────────
feature_analysis = {}

def analyze_feature(name, description, source, vals, y, expected_importance):
    """Analyze a single feature."""
    cancelled_vals = vals[y == 1]
    arrived_vals = vals[y == 0]
    
    mean_diff = cancelled_vals.mean() - arrived_vals.mean()
    # Simple signal score: |mean_diff| / (arrived_vals.std() + 1e-8)
    signal_score = abs(mean_diff) / (arrived_vals.std() + 1e-8)
    
    return {
        'name': name,
        'description': description,
        'source': source,
        'data_type': 'binary' if vals.nunique() <= 2 else 'continuous',
        'count': len(vals),
        'unique': vals.nunique(),
        'missing': int(vals.isna().sum()),
        'min': vals.min(),
        'max': vals.max(),
        'mean': vals.mean(),
        'std': vals.std(),
        'zeros': int((vals == 0).sum()),
        'zero_pct': (vals == 0).mean() * 100,
        'cancelled_mean': cancelled_vals.mean() if len(cancelled_vals) > 0 else None,
        'arrived_mean': arrived_vals.mean() if len(arrived_vals) > 0 else None,
        'cancelled_count': len(cancelled_vals),
        'arrived_count': len(arrived_vals),
        'mean_diff': mean_diff,
        'signal_score': signal_score,
        'signal_label': 'SIGNAL' if signal_score > 1.0 else ('WEAK' if signal_score > 0.2 else 'NOISE'),
        'expected_importance': expected_importance,
    }

# Define each feature's metadata
features_meta = [
    ('inbound_delay', 'Inbound aircraft delay minutes (from AeroDataBox)', 'flightStatus.inboundDelayMinutes', 'MEDIUM — should catch cascading delays'),
    ('current_delay', 'Current delay minutes (from AeroDataBox)', 'flightStatus.delayMinutes', 'HIGH — direct operational disruption signal'),
    ('atc_avg_delay', 'Max of NAS origin/destination avg delay (from FAA NAS API)', 'nasOrigin.avgDelayMinutes / nasDestination.avgDelayMinutes', 'MEDIUM — ATC congestion indicator'),
    ('origin_flight_cat', 'Origin flight category (VFR=0, MVFR=1, IFR=2, LIFR=3)', 'originWeather.flightCategory', 'LOW — weather ceiling/visibility proxy'),
    ('origin_thunderstorm', 'Thunderstorm at origin (binary)', 'originWeather.hasThunderstorm', 'LOW — severe weather indicator'),
    ('origin_freezing', 'Freezing conditions at origin (binary)', 'originWeather.hasFreezing', 'LOW — deicing/safety indicator'),
    ('origin_wind_speed', 'Wind speed at origin (knots)', 'originWeather.windSpeedKt', 'LOW — crosswind/operational impact'),
    ('origin_gust_speed', 'Gust speed at origin (knots)', 'originWeather.gustSpeedKt', 'LOW — gust impact on operations'),
    ('origin_visibility', 'Visibility at origin (miles)', 'originWeather.visibilityMiles', 'LOW — visibility impact on takeoff'),
    ('origin_ceiling', 'Ceiling at origin (feet)', 'originWeather.ceilingFt', 'LOW — cloud ceiling impact'),
    ('dest_flight_cat', 'Destination flight category (VFR=0 to LIFR=3)', 'destinationWeather.flightCategory', 'MEDIUM — destination weather affects landing'),
    ('dest_thunderstorm', 'Thunderstorm at destination (binary)', 'destinationWeather.hasThunderstorm', 'LOW — severe weather at arrival'),
    ('dest_freezing', 'Freezing at destination (binary)', 'destinationWeather.hasFreezing', 'LOW — icing at arrival'),
    ('carrier_cancel_rate', 'Carrier cancellation rate over last 24h', 'carrierHealth.cancellationRate24h', 'HIGH — carrier operational health'),
    ('carrier_avg_delay', 'Carrier average delay over last 24h', 'carrierHealth.avgDelay24h', 'MEDIUM — carrier delay trend'),
    ('carrier_sample_size', 'Number of flights sampled for carrier stats', 'carrierHealth.sampleSize', 'LOW — confidence weight for carrier features'),
    ('departure_hour', 'Scheduled departure hour (0-23)', 'flightStatus.departureTime / departure_time', 'MEDIUM — time-of-day patterns'),
    ('day_of_week', 'Day of week (JS: 0=Sun, 6=Sat)', 'flightStatus.departureTime / departure_date', 'MEDIUM — day-of-week patterns'),
    ('month', 'Month (1-12)', 'flightStatus.departureTime / departure_date', 'LOW — seasonal patterns'),
    ('is_weekend', 'Is weekend flag (Sat/Sun)', 'derived from day_of_week', 'LOW — weekend operational patterns'),
    ('is_rush_hour', 'Is rush hour flag (7-9, 16-18)', 'derived from departure_hour', 'LOW — peak congestion proxy'),
    ('is_summer', 'Is summer flag (Jun-Aug)', 'derived from month', 'LOW — summer weather proxy'),
    ('delay_delta_30m', 'Change in delay since last monitoring cycle', 'computed: current_delay - previous_delay', 'MEDIUM — delay trend (journey tracking)'),
    ('delay_rolling_avg_4', 'Rolling average of delay over last 4 cycles', 'computed: avg of last 4 current_delay values', 'MEDIUM — delay smoothing'),
    ('checkpoint_number', 'Which monitoring cycle in this flight\'s journey', 'computed: per-flight cycle counter', 'LOW — monitoring intensity proxy'),
]

for name, desc, source, importance in features_meta:
    feature_analysis[name] = analyze_feature(
        name, desc, source,
        X_prod_filtered[name], y_final, importance
    )

# Display as table
analysis_df = pd.DataFrame(feature_analysis.values())
display_cols = ['name', 'signal_label', 'signal_score', 'mean_diff', 'zeros', 'zero_pct', 'mean', 'cancelled_mean', 'arrived_mean', 'expected_importance']
print('--- Feature Analysis Summary ---')
print(analysis_df[display_cols].to_string(index=False))

print(f'\n\n--- Signal Classification ---')
for label in ['SIGNAL', 'WEAK', 'NOISE']:
    subset = analysis_df[analysis_df['signal_label'] == label]
    print(f'  {label}: {list(subset["name"])}')

--- Feature Analysis Summary ---
               name signal_label  signal_score     mean_diff  zeros   zero_pct         mean  cancelled_mean  arrived_mean                          expected_importance
      inbound_delay        NOISE      0.016655     -0.024962   7226  99.972330     0.024903        0.000000      0.024962       MEDIUM — should catch cascading delays
      current_delay        NOISE      0.016655     -0.024962   7226  99.972330     0.024903        0.000000      0.024962  HIGH — direct operational disruption signal
      atc_avg_delay       SIGNAL      1.376522     84.333012   6306  87.244051    18.982983      103.117647     18.784635            MEDIUM — ATC congestion indicator
  origin_flight_cat         WEAK      0.245314     -0.096079   6176  85.445490     0.154676        0.058824      0.154902       LOW — weather ceiling/visibility proxy
origin_thunderstorm        NOISE      0.179451     -0.031202   7003  96.887106     0.031129        0.000000      0.031202           

In [14]:
# ──────────────────────────────────────────────
# CELL: Detailed per-feature drill-down
# ──────────────────────────────────────────────
for name, meta in feature_analysis.items():
    print(f'{"="*60}')
    print(f'Feature: {name}')
    print(f'  Data source: {meta["source"]}')
    print(f'  Description: {meta["description"]}')
    print(f'  Expected importance: {meta["expected_importance"]}')
    print(f'  Type: {meta["data_type"]}, Unique: {meta["unique"]}')
    print(f'  Range: [{meta["min"]:.4f}, {meta["max"]:.4f}]')
    print(f'  Mean: {meta["mean"]:.4f}, Std: {meta["std"]:.4f}')
    print(f'  Zeros: {meta["zeros"]}/{meta["count"]} ({meta["zero_pct"]:.1f}%)')
    print(f'  Cancelled mean: {meta["cancelled_mean"]}')
    print(f'  Arrived mean:   {meta["arrived_mean"]}')
    print(f'  Mean difference: {meta["mean_diff"]:.4f}')
    print(f'  Signal score: {meta["signal_score"]:.4f} [{meta["signal_label"]}]')
    
    # Data quality notes
    issues = []
    if meta['zero_pct'] > 99:
        issues.append(f'WARNING: {meta["zero_pct"]:.1f}% zeros — effectively dead feature')
    if meta['unique'] <= 1:
        issues.append('CRITICAL: Zero variance — will be ignored by tree models')
    if abs(meta['mean_diff']) < 0.01 and meta['mean'] > 0:
        issues.append('Very small mean diff relative to scale — low predictive power')
    if 'atc_avg_delay' in name:
        issues.append('NOTE: featureExtractor.ts reads nasOrigin/nasDestination, but riskScorer.ts provides nasStatus (mismatch!)')
    if name == 'carrier_cancel_rate':
        issues.append('NOTE: T.ipynb/T2.ipynb divide by 100 (BUG!). Correct value is 0-1 rate.')
    if 'origin_visibility' in name:
        issues.append(f'Range is tight: {meta["min"]}-{meta["max"]} — most values near 10')
    if 'dest_flight_cat' in name:
        issues.append('Destination weather only has 3 fields stored (flightCategory, hasThunderstorm, hasFreezing)')
    
    if issues:
        print(f'  Data quality issues:')
        for issue in issues:
            print(f'    - {issue}')
    print()

Feature: inbound_delay
  Data source: flightStatus.inboundDelayMinutes
  Description: Inbound aircraft delay minutes (from AeroDataBox)
  Expected importance: MEDIUM — should catch cascading delays
  Type: binary, Unique: 2
  Range: [0.0000, 90.0000]
  Mean: 0.0249, Std: 1.4970
  Zeros: 7226/7228 (100.0%)
  Cancelled mean: 0.0
  Arrived mean:   0.024961863819165164
  Mean difference: -0.0250
  Signal score: 0.0167 [NOISE]
  Data quality issues:
    - WARNING: 100.0% zeros — effectively dead feature

Feature: current_delay
  Data source: flightStatus.delayMinutes
  Description: Current delay minutes (from AeroDataBox)
  Expected importance: HIGH — direct operational disruption signal
  Type: binary, Unique: 2
  Range: [0.0000, 90.0000]
  Mean: 0.0249, Std: 1.4970
  Zeros: 7226/7228 (100.0%)
  Cancelled mean: 0.0
  Arrived mean:   0.024961863819165164
  Mean difference: -0.0250
  Signal score: 0.0167 [NOISE]
  Data quality issues:
    - WARNING: 100.0% zeros — effectively dead feature

F

In [15]:
# ──────────────────────────────────────────────
# CELL: Temporal feature deep-dive
# ──────────────────────────────────────────────
print('--- Journey Tracking Analysis ---')

cp = X_prod_filtered['checkpoint_number']
dd = X_prod_filtered['delay_delta_30m']
ra = X_prod_filtered['delay_rolling_avg_4']

print(f'Checkpoint number distribution:')
print(f'  Range: {cp.min()} to {cp.max()}')
print(f'  Mean: {cp.mean():.1f}')
print(f'  The journey tracking sequence exists — each flight gets numbered cycles.')
print(f'  This feature is NOT dead (100% non-zero, good variance).')

print(f'\ndelay_delta_30m:')
print(f'  Non-zero: {(dd != 0).sum()}/{len(dd)} ({((dd != 0).mean()*100):.2f}%)')
print(f'  Why? delayMinutes is 0 in 99.9% of rows, so delta is always 0.')
print(f'  The tracking logic works, but the INPUT to track has no signal.')

print(f'\ndelay_rolling_avg_4:')
print(f'  Non-zero: {(ra != 0).sum()}/{len(ra)} ({((ra != 0).mean()*100):.2f}%)')
print(f'  Same problem — with no delay values, rolling avg stays 0.')

print(f'\n--- Root Cause ---')
print(f'This dataset spans 2026-05-17 to 2026-06-10 (3 weeks).')
print(f'AeroDataBox returned delayMinutes=0 for almost all flights.')
print(f'Two possibilities:')
print(f'  1. The flights genuinely had no delays (unlikely for 478 flights over 3 weeks)')
print(f'  2. The AeroDataBox API call didn\'t return delay data (most likely)')
print(f'  Check monitor.ts / flightStatus.ts to see if delayMinutes is correctly fetched')

--- Journey Tracking Analysis ---
Checkpoint number distribution:
  Range: 1 to 68
  Mean: 13.1
  The journey tracking sequence exists — each flight gets numbered cycles.
  This feature is NOT dead (100% non-zero, good variance).

delay_delta_30m:
  Non-zero: 4/7228 (0.06%)
  Why? delayMinutes is 0 in 99.9% of rows, so delta is always 0.
  The tracking logic works, but the INPUT to track has no signal.

delay_rolling_avg_4:
  Non-zero: 8/7228 (0.11%)
  Same problem — with no delay values, rolling avg stays 0.

--- Root Cause ---
This dataset spans 2026-05-17 to 2026-06-10 (3 weeks).
AeroDataBox returned delayMinutes=0 for almost all flights.
Two possibilities:
  1. The flights genuinely had no delays (unlikely for 478 flights over 3 weeks)
  2. The AeroDataBox API call didn't return delay data (most likely)
  Check monitor.ts / flightStatus.ts to see if delayMinutes is correctly fetched


In [16]:
# ──────────────────────────────────────────────
# CELL: Production inference mismatch summary
# ──────────────────────────────────────────────
print(f'{"="*60}')
print('CRITICAL: Feature set mismatch between training and production')
print(f'{"="*60}\n')

notebook_features = set([
    'carrier_cancel_rate', 'carrier_sample_size', 'carrier_avg_delay',
    'mins_until_departure', 'departure_hour', 'departure_minute',
    'day_of_month', 'month', 'day_of_week', 'is_weekend',
    'origin_ceiling_ft', 'origin_visibility_miles', 'origin_wind_speed_kt',
    'origin_flight_category', 'destination_visibility_miles',
    'destination_flight_category', 'has_thunderstorm', 'has_freezing',
    'gust_speed_kt', 'delay_delta_30m', 'delay_rolling_avg_4',
    'atc_delay', 'inbound_delay', 'checkpoint_number',
    'mins_until_dep_sq', 'mins_until_dep_sqrt'
])

production_features = set(expected_order)

print('Features in T.ipynb/T2.ipynb but NOT in featureExtractor.ts:')
extra_in_notebook = notebook_features - production_features
for f in sorted(extra_in_notebook):
    print(f'  ✗ {f}')
    
print(f'\nFeatures in featureExtractor.ts but NOT in T.ipynb/T2.ipynb:')
missing_in_notebook = production_features - notebook_features
for f in sorted(missing_in_notebook):
    print(f'  ✗ {f}')

print(f'\nFeatures with SAME NAME but DIFFERENT SOURCE:')
print(f'  atc_avg_delay (production) vs atc_delay (notebook)')
print(f'    Production: nasOrigin/nasDestination.avgDelayMinutes (FAA NAS API)')
print(f'    Notebook:   flightStatus.delayMinutes (AeroDataBox API)')
print(f'  These are COMPLETELY DIFFERENT data sources!')

print(f'\nFeatures with DIFFERENT SCALING:')
print(f'  carrier_cancel_rate:')
print(f'    Production: cancellationRate24h AS-IS (0-1 rate, e.g. 0.15 = 15%)')
print(f'    Notebook:   cancels / 100 (0-0.01, e.g. 0.0015 = 0.15%)')
print(f'    100x scale difference!')

CRITICAL: Feature set mismatch between training and production

Features in T.ipynb/T2.ipynb but NOT in featureExtractor.ts:
  ✗ atc_delay
  ✗ day_of_month
  ✗ departure_minute
  ✗ destination_flight_category
  ✗ destination_visibility_miles
  ✗ gust_speed_kt
  ✗ has_freezing
  ✗ has_thunderstorm
  ✗ mins_until_dep_sq
  ✗ mins_until_dep_sqrt
  ✗ mins_until_departure
  ✗ origin_ceiling_ft
  ✗ origin_flight_category
  ✗ origin_visibility_miles
  ✗ origin_wind_speed_kt

Features in featureExtractor.ts but NOT in T.ipynb/T2.ipynb:
  ✗ atc_avg_delay
  ✗ current_delay
  ✗ dest_flight_cat
  ✗ dest_freezing
  ✗ dest_thunderstorm
  ✗ is_rush_hour
  ✗ is_summer
  ✗ origin_ceiling
  ✗ origin_flight_cat
  ✗ origin_freezing
  ✗ origin_gust_speed
  ✗ origin_thunderstorm
  ✗ origin_visibility
  ✗ origin_wind_speed

Features with SAME NAME but DIFFERENT SOURCE:
  atc_avg_delay (production) vs atc_delay (notebook)
    Production: nasOrigin/nasDestination.avgDelayMinutes (FAA NAS API)
    Notebook:   fl

In [17]:
# ──────────────────────────────────────────────
# CELL: NAS feature — inference pipeline bug
# ──────────────────────────────────────────────
print(f'{"="*60}')
print('NAS FEATURE MISMATCH (atc_avg_delay)')
print(f'{"="*60}\n')
print('featureExtractor.ts reads: sig.nasOrigin.avgDelayMinutes and sig.nasDestination.avgDelayMinutes')
print()
print('But riskScorer.ts creates signalsForML (line 350-386) with:')
print('  nasStatus: { hasGroundStop, hasGroundDelay, avgDelayMinutes }')
print('  (NOT nasOrigin/nasDestination)')
print()
print('This means during production inference:')
print('  1. riskScorer.ts builds MLInput with signals.nasStatus')
print('  2. featureExtractor.ts reads signals.nasOrigin and signals.nasDestination')
print('  3. Both are {} → atc_avg_delay ALWAYS = 0')
print()
print('The model was trained with non-zero atc_avg_delay values')
print('but inference will always feed it 0. This breaks the model.')
print()
print('Fix: riskScorer.ts should set signals.nasOrigin and signals.nasDestination')
print('     OR featureExtractor.ts should read sig.nasStatus')

NAS FEATURE MISMATCH (atc_avg_delay)

featureExtractor.ts reads: sig.nasOrigin.avgDelayMinutes and sig.nasDestination.avgDelayMinutes

But riskScorer.ts creates signalsForML (line 350-386) with:
  nasStatus: { hasGroundStop, hasGroundDelay, avgDelayMinutes }
  (NOT nasOrigin/nasDestination)

This means during production inference:
  1. riskScorer.ts builds MLInput with signals.nasStatus
  2. featureExtractor.ts reads signals.nasOrigin and signals.nasDestination
  3. Both are {} → atc_avg_delay ALWAYS = 0

The model was trained with non-zero atc_avg_delay values
but inference will always feed it 0. This breaks the model.

Fix: riskScorer.ts should set signals.nasOrigin and signals.nasDestination
     OR featureExtractor.ts should read sig.nasStatus


In [18]:
# ──────────────────────────────────────────────
# CELL: Destination weather — inference vs training mismatch
# ──────────────────────────────────────────────
print(f'{"="*60}')
print('DESTINATION WEATHER: training vs inference')
print(f'{"="*60}\n')
print('Training data (from DB JSONB, monitor.ts lines 105-109):')
print('  destinationWeather has ONLY: flightCategory, hasThunderstorm, hasFreezing')
print()
print('Inference (from riskScorer.ts signalsForML lines 360-368):')
print('  destinationWeather has: flightCategory, hasThunderstorm, hasFreezing,')
print('    windSpeedKt, gustSpeedKt, visibilityMiles, ceilingFt')
print()
print('featureExtractor.ts only reads the 3 fields that are stored, so this is OK.')
print('But if we ever want to add more destination weather features,')
print('we must update monitor.ts to store them in the JSONB.')

DESTINATION WEATHER: training vs inference

Training data (from DB JSONB, monitor.ts lines 105-109):
  destinationWeather has ONLY: flightCategory, hasThunderstorm, hasFreezing

Inference (from riskScorer.ts signalsForML lines 360-368):
  destinationWeather has: flightCategory, hasThunderstorm, hasFreezing,
    windSpeedKt, gustSpeedKt, visibilityMiles, ceilingFt

featureExtractor.ts only reads the 3 fields that are stored, so this is OK.
But if we ever want to add more destination weather features,
we must update monitor.ts to store them in the JSONB.


In [19]:
# ──────────────────────────────────────────────
# CELL: Final summary
# ──────────────────────────────────────────────
print(f'{"="*60}')
print('SUMMARY OF FINDINGS')
print(f'{"="*60}\n')

print('1. FEATURE SET MISMATCH (CRITICAL)')
print('   T.ipynb/T2.ipynb extract 26 features that do NOT match')
print('   featureExtractor.ts (25 features). The saved model is incompatible.')
print()
print('2. CARRIER CANCEL RATE SCALING BUG (CRITICAL)')
print('   Notebooks divide cancellationRate24h by 100 (wrong — it is already 0-1).')
print('   featureExtractor.ts uses it as-is. 100x scale difference.')
print()
print('3. NAS FEATURE SOURCE BUG (CRITICAL)')
print('   featureExtractor.ts reads nasOrigin/nasDestination.avgDelayMinutes.')
print('   riskScorer.ts provides nasStatus.avgDelayMinutes at inference.')
print('   atc_avg_delay will ALWAYS be 0 in production.')
print()
print('4. TEMPORAL FEATURES ARE DEAD (DATA QUALITY)')
print('   delay_delta_30m: 99.9% zeros')
print('   delay_rolling_avg_4: 99.9% zeros')
print('   The journey tracking logic works but AeroDataBox delayMinutes is 0.')
print()
print('5. DESTINATION WEATHER PARTIALLY STORED')
print('   monitor.ts only stores 3/7 destinationWeather fields.')
print('   featureExtractor.ts correctly only uses stored fields.')
print('   Notebooks incorrectly extract visibility (always 10).')
print()
print('6. IMBALANCE & WINDOW')
print('   Only 17 positive cycles (0.24%) after filter.')
print('   3-week window is too short for meaningful carrier stats.')
print('   carrier_cancel_rate has 0 mean because most carriers had 0 cancellations.')

SUMMARY OF FINDINGS

1. FEATURE SET MISMATCH (CRITICAL)
   T.ipynb/T2.ipynb extract 26 features that do NOT match
   featureExtractor.ts (25 features). The saved model is incompatible.

2. CARRIER CANCEL RATE SCALING BUG (CRITICAL)
   Notebooks divide cancellationRate24h by 100 (wrong — it is already 0-1).
   featureExtractor.ts uses it as-is. 100x scale difference.

3. NAS FEATURE SOURCE BUG (CRITICAL)
   featureExtractor.ts reads nasOrigin/nasDestination.avgDelayMinutes.
   riskScorer.ts provides nasStatus.avgDelayMinutes at inference.
   atc_avg_delay will ALWAYS be 0 in production.

4. TEMPORAL FEATURES ARE DEAD (DATA QUALITY)
   delay_delta_30m: 99.9% zeros
   delay_rolling_avg_4: 99.9% zeros
   The journey tracking logic works but AeroDataBox delayMinutes is 0.

5. DESTINATION WEATHER PARTIALLY STORED
   monitor.ts only stores 3/7 destinationWeather fields.
   featureExtractor.ts correctly only uses stored fields.
   Notebooks incorrectly extract visibility (always 10).

6. IMBAL

---
## Conclusion

### The training pipeline has **3 critical bugs** that make the saved model incompatible with production:

1. **Wrong feature set** — 26 notebook features vs 25 production features
2. **Wrong scaling** — carrier_cancel_rate divided by 100 in notebooks
3. **Wrong source** — atc_delay from flightStatus vs atc_avg_delay from NAS

### Data quality issues:
- Temporal features are dead (99.9% zeros) — AeroDataBox returns no delay data
- Only 0.24% positive examples — insufficient for training
- Destination weather is partially stored

### Inference pipeline bug:
- riskScorer.ts provides `nasStatus` but featureExtractor.ts reads `nasOrigin`/`nasDestination`